## 1. Imports and Setup
We start by importing the necessary modules, setting up the test image, keywords, and initializing models via the `model_manager`.


In [ ]:
import logging
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Import from the application backend
from backend.models import model_manager
from backend.image_processor import (
    _extract_objects,
    _link_overlap_partners,
    _complete_overlapping_objects,
    _apply_pair_decisions,
    _build_reconstruction_masks
)
from backend.core.occlusion import assign_pair_roles
from backend.core.helpers import _calc_kernel_size

# Setup logging to see output from the orchestrator
logging.basicConfig(level=logging.INFO)

# Setup a test image (Update this path to a real image with overlapping objects)
image_path = "tests/test_data/sample.jpg" 
try:
    image = Image.open(image_path).convert("RGB")
except FileNotFoundError:
    # Fallback to creating a dummy RGB image if the path does not exist
    image = Image.new('RGB', (512, 512), color=(200, 200, 200))
    print(f"Warning: Test image not found at {image_path}. Using a blank 512x512 image.")

# Keywords expected to trigger cross-class overlap
keywords = ["person", "car"]

# Visualize the original image
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title("Original Test Image")
plt.axis('off')
plt.show()


## 2. Object Extraction (SAM3 Grouping)
Extract objects using the segmentation model. The results are grouped into `DetectedObject` records, which include modal masks and tight bounding boxes for each semantic class.


In [ ]:
# Run segmentation
objects = _extract_objects(image, keywords)
print(f"\nExtracted {len(objects)} grouped objects.")

# Visualize the modal masks
if objects:
    fig, axes = plt.subplots(1, len(objects), figsize=(5 * len(objects), 5))
    if len(objects) == 1:
        axes = [axes]
    
    for ax, obj in zip(axes, objects):
        ax.imshow(obj.modal_mask, cmap='gray')
        ax.set_title(f"{obj.display_label}\nBBox: {obj.bbox}")
        ax.axis('off')
    plt.show()
else:
    print("No objects detected. The pipeline will skip further steps.")


## 3. Cross-Class Overlap Graph
Evaluate the bounding boxes to find overlaps across different semantic classes. This triggers whether an object is a candidate for amodal completion.

In [ ]:
overlap_pairs = _link_overlap_partners(objects)
print(f"Found {len(overlap_pairs)} cross-class bounding-box overlap pairs.")

for obj in objects:
    print(f"Object {obj.object_id} ({obj.display_label}) overlaps with partner IDs: {obj.overlap_partner_ids}")


## 4. Conditional Amodal Completion
Run the batch completion model (via SDAmodal and shared DIFT extraction) ONLY for objects involved in cross-class overlaps. This populates `amodal_mask` and `completion_hole_mask`.


In [ ]:
_complete_overlapping_objects(image, objects)

# Visualize amodal masks and the generated completion holes
completed_objects = [obj for obj in objects if obj.amodal_mask is not None]

if completed_objects:
    fig, axes = plt.subplots(2, len(completed_objects), figsize=(5 * len(completed_objects), 10))
    if len(completed_objects) == 1:
        axes = np.expand_dims(axes, axis=1)

    for i, obj in enumerate(completed_objects):
        # Amodal mask
        axes[0, i].imshow(obj.amodal_mask, cmap='gray')
        axes[0, i].set_title(f"{obj.display_label}\nAmodal Mask")
        axes[0, i].axis('off')
        
        # Completion hole mask
        axes[1, i].imshow(obj.completion_hole_mask, cmap='gray')
        axes[1, i].set_title(f"{obj.display_label}\nHole Area: {obj.completion_hole_area} px")
        axes[1, i].axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("No objects required amodal completion.")


## 5. Pairwise Role Decisions
Compare completion-hole areas per overlap edge. The object with the larger hole area is chosen as the occluded object, while the other acts as the occluder.


In [ ]:
# Gather the calculated hole areas for decision making
hole_areas = {
    obj.object_id: obj.completion_hole_area 
    for obj in objects if obj.completion_hole_area is not None
}

pair_decisions = assign_pair_roles(overlap_pairs, hole_areas)

print("Decisions:")
for pd in pair_decisions:
    if pd.ambiguous:
        print(f"- Pair ({pd.first_id}, {pd.second_id}) is ambiguous. No roles assigned.")
    else:
        print(f"- Object {pd.occluder_id} occludes Object {pd.occluded_id}.")

# Apply decisions back to the detected objects
_apply_pair_decisions(objects, pair_decisions)

print("\nFinal Occluder Assignments:")
for obj in objects:
    print(f"- {obj.object_id} ({obj.display_label}) is occluded by: {obj.occluder_ids}")


## 6. Reconstruction Masks Construction
Finally, build the constrained reconstruction mask for any object with assigned occluders. This mask unites the completion hole and the relevant portions of the occluders.


In [ ]:
image_np = np.asarray(image, dtype=np.uint8)
kernel_size = _calc_kernel_size(image_np)

_build_reconstruction_masks(objects, kernel_size)

# Visualize reconstruction masks
reconstruction_objects = [obj for obj in objects if obj.reconstruction_mask is not None]

if reconstruction_objects:
    fig, axes = plt.subplots(1, len(reconstruction_objects), figsize=(5 * len(reconstruction_objects), 5))
    if len(reconstruction_objects) == 1:
        axes = [axes]
        
    for ax, obj in zip(axes, reconstruction_objects):
        ax.imshow(obj.reconstruction_mask, cmap='gray')
        ax.set_title(f"{obj.display_label}\nReconstruction Mask")
        ax.axis('off')
    plt.show()
else:
    print("No objects require reconstruction.")


## End of Implemented Pipeline
According to `coding_progress.md`, the implementation successfully reaches this reconstruction-mask construction block. 

Subsequent steps like hidden RGB reconstruction via inpainting, refactored BiRefNet per-object inputs, and final object-layer extraction are marked as pending or to-do, and are intentionally omitted from this test script.
